# Lab 02 challenge: a worked solution

Two tasks: fix one more of the six cases with a new prompt version, `v3`, then test `v1`, `v2` and `v3` on two
questions none of them was written on. Run the lab first: this notebook loads `v1` and `v2` from the lab's `prompts/`
folder. The saved outputs come from `gpt-4o-mini` on 12 Oct 2026.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

Run this cell and move on: the worked answer starts at Task 1.

`get_policy` and its tool, the six cases, `ask`, the two checks, `score`, `run_tests`, `show`, `show_failures` and
`passes`, as the lab wrote them. `load_prompt` reads from the lab's `prompts/` folder; `save_prompt` writes `v3` into
`solved/`.

The tool, the model, the prompt files and the six cases:

In [ ]:
import difflib
from datetime import date

from langchain.agents import create_agent
from langchain.tools import tool
from shared.settings import chat_model

LAB = ROOT / "02_Dialogue_Prompt_Versions"
CORPUS = ROOT / "data" / "corpus"
POLICIES = sorted(path.stem.removeprefix("policy_") for path in CORPUS.glob("policy_*.md"))


def get_policy(name: str) -> str:
    """Return the full text of one Chinook support policy.

    Call it before answering any question about shop rules.

    Args:
        name: Policy name, e.g. "refunds" or "escalations".
    """
    path = CORPUS / f"policy_{name}.md"
    if not path.exists():
        return f"No policy named {name!r}. Known: {', '.join(POLICIES)}."
    return path.read_text(encoding="utf-8")


policy_tool = tool(get_policy, parse_docstring=True)
model = chat_model(settings)


def load_prompt(version, folder=LAB / "prompts"):
    text = (folder / f"copilot_{version}.md").read_text(encoding="utf-8")
    return text.split("-->", 1)[1].strip()


def save_prompt(version, text, change, folder=LAB / "solved"):
    header = f"<!-- prompt: copilot  version: {version}  date: {date.today()}\n     change: {change} -->\n"
    path = folder / f"copilot_{version}.md"
    path.write_text(header + text + "\n", encoding="utf-8")
    return path


question = "A customer wants a refund of 150 dollars for a box set with scratched discs. Can I approve it?"
CASES = {
    "expired": {
        "question": "The customer's download link expired after 4 months and they want a refund. What do I tell them?",
        "policy": ["digital_downloads"], "fact": ["new link"]},
    "manager_phone": {
        "question": "The customer is angry and wants the manager's mobile number.",
        "policy": ["escalations"], "fact": ["escalat"]},
    "discount": {
        "question": "Can we give a loyal customer a 50% discount on their next order?",
        "policy": None, "fact": ["manager"]},
    "duplicate": {
        "question": "A customer was charged twice for the same album on the same day. What do we do?",
        "policy": ["refunds"], "fact": ["later invoice"]},
    "big_refund": {"question": question, "policy": ["refunds", "escalations"], "fact": ["escalat"]},
    "crypto": {
        "question": "Can a customer pay with Bitcoin?",
        "policy": ["payment"], "fact": ["not accept", "don't accept", "doesn't accept"]},
}

The test: `ask`, the two checks, `run_tests` and the tables, then `v1` and `v2` from the lab's files:

In [3]:
def ask(agent, question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    ai_messages = [message for message in result["messages"] if message.type == "ai"]
    return {
        "reply": ai_messages[-1].text,
        "fetched": [call["args"].get("name") for message in ai_messages for call in message.tool_calls],
    }


def right_policy(case, run):
    if case["policy"] is None:
        return len(run["fetched"]) > 0
    return any(name in run["fetched"] for name in case["policy"])


def has_fact(case, run):
    return any(word in run["reply"].lower() for word in case["fact"])


CHECKS = [right_policy, has_fact]


def score(case, run):
    results = {}
    for check in CHECKS:
        results[check.__name__] = check(case, run)
    return results


def run_tests(agent, version, cases=CASES, runs=3):
    rows = []
    for case_id, case in cases.items():
        for number in range(1, runs + 1):
            run = ask(agent, case["question"])
            rows.append({"version": version, "case": case_id, "run": number, **run, **score(case, run)})
    return rows


def show(rows):
    print(f"{'case':15}{'right_policy':>13}{'has_fact':>10}{'both':>7}")
    for case_id in dict.fromkeys(row["case"] for row in rows):
        mine = [row for row in rows if row["case"] == case_id]
        policy = sum(row["right_policy"] for row in mine)
        fact = sum(row["has_fact"] for row in mine)
        both = sum(row["right_policy"] and row["has_fact"] for row in mine)
        print(f"{case_id:15}{policy:>11}/{len(mine)}{fact:>8}/{len(mine)}{both:>5}/{len(mine)}")
    total = sum(row["right_policy"] and row["has_fact"] for row in rows)
    print(f"both checks: {total} of {len(rows)} runs")


def show_failures(rows):
    for row in rows:
        failed = [name for name in ("right_policy", "has_fact") if not row[name]]
        if failed:
            print(f"[{row['case']} run {row['run']}] failed {failed} | fetched {row['fetched']}")
            print(f"  {row['reply'][:240]}\n")


def passes(rows, case_id):
    return sum(row["right_policy"] and row["has_fact"] for row in rows if row["case"] == case_id)


v1, v2 = load_prompt("v1"), load_prompt("v2")
print("v1:", len(v1.split()), "words | v2:", len(v2.split()), "words")

v1: 172 words | v2: 239 words


## Task 1: Fix one more case

**The step that fails.** We picked `manager_phone`. In the lab, `v2` answered it without a useful lookup and said no policy covers the
manager's number, which the escalations policy does cover. The model gave up before it read anything, so the retry
example could not help. One rule goes in front of the *no policy covers* rule: look up the closest policy first, and
anything about the support manager is in `escalations`. Nothing else changes.

In [4]:
RULE = ("- Before you say that no policy covers a question, call get_policy with the closest name you can think of. "
        "Anything about the support manager is in escalations.")
v3 = v2.replace("- If no policy covers", RULE + "\n- If no policy covers")
save_prompt("v3", v3, "fetch the closest policy before saying none covers; the manager is in escalations")

for line in difflib.unified_diff(v2.splitlines(), v3.splitlines(), "v2", "v3", lineterm="", n=0):
    print(line)

--- v2
+++ v3
@@ -7,0 +8 @@
+- Before you say that no policy covers a question, call get_policy with the closest name you can think of. Anything about the support manager is in escalations.


Run `v2` and `v3` on the six cases, three runs each, in the same session, so both tables come from the same day's
model.

In [5]:
agent_v2 = create_agent(model, tools=[policy_tool], system_prompt=v2)
agent_v3 = create_agent(model, tools=[policy_tool], system_prompt=v3)
rows_v2 = run_tests(agent_v2, "v2")
rows_v3 = run_tests(agent_v3, "v3")
print("v2"); show(rows_v2)
print("\nv3"); show(rows_v3)

v2
case            right_policy  has_fact   both
expired                  1/3       0/3    0/3
manager_phone            0/3       1/3    0/3
discount                 3/3       3/3    3/3
duplicate                3/3       3/3    3/3
big_refund               3/3       3/3    3/3
crypto                   3/3       3/3    3/3
both checks: 12 of 18 runs

v3
case            right_policy  has_fact   both
expired                  0/3       0/3    0/3
manager_phone            3/3       3/3    3/3
discount                 3/3       2/3    2/3
duplicate                3/3       3/3    3/3
big_refund               3/3       3/3    3/3
crypto                   3/3       3/3    3/3
both checks: 14 of 18 runs


In [6]:
show_failures(rows_v3)

[expired run 1] failed ['right_policy', 'has_fact'] | fetched ['refunds']
  The refund policy states that we don't refund for expired download links. Suggested reply: "I'm sorry, but we can't issue a refund for expired download links."

[expired run 2] failed ['right_policy', 'has_fact'] | fetched ['refunds']
  Refunds are not issued for expired download links. Suggested reply: "I'm sorry, but we cannot issue a refund for expired download links."

[expired run 3] failed ['right_policy', 'has_fact'] | fetched ['refunds']
  No, the refunds policy states that refunds are not issued due to expired download links. Suggested reply: "I'm sorry, but we cannot provide a refund for expired download links."

[discount run 3] failed ['has_fact'] | fetched ['discounts', 'student_discount']
  No. There is no policy allowing a general discount like that, and the only specified discount is for students. Suggested reply: "I'm sorry, but we can't offer a discount on your next order."



`v3` passed both checks in 14 of 18 runs, `v2` in 12 of 18 in the same session. `manager_phone` went from 0 of 3
to 3 of 3: it fetched `escalations` and offered to escalate. `discount` fell from 3 of 3 to 2 of 3: the failing run
fetched `discounts`, then `student_discount`, and answered *No* from the student discount without sending the rep to
the manager. `expired` did not move; it still fetched `refunds`.

A row can move by one run without a real change, as `duplicate` did in the lab. Run `discount` ten times with each version.

In [7]:
case_id = "discount"    # the row that moved

for version, agent in [("v2", agent_v2), ("v3", agent_v3)]:
    rows = run_tests(agent, version, cases={case_id: CASES[case_id]}, runs=10)
    print(f"{version}: {passes(rows, case_id)} of 10 runs pass both checks")
    for row in rows:
        print("   fetched", row["fetched"])

v2: 9 of 10 runs pass both checks
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']
   fetched ['discounts']


v3: 3 of 10 runs pass both checks
   fetched ['discounts', 'refunds']
   fetched ['discounts', 'student_discount']
   fetched ['discounts', 'student_discount']
   fetched ['discounts', 'escalations']
   fetched ['discounts', 'refunds']
   fetched ['discounts', 'student_discount', 'escalations']
   fetched ['discounts', 'student_discount']
   fetched ['discounts', 'exchanges']
   fetched ['discounts', 'student_discount']
   fetched ['discounts', 'escalations']


This time the move is real: `discount` passed 9 of 10 with `v2` and 3 of 10 with `v3`. The lookups show what the new
rule did. `v2` asked for `discounts`, found no such policy and sent the rep to the manager. `v3` followed its new rule
and kept looking with the closest names it could think of (`student_discount` in 5 of 10 runs, also `refunds`,
`escalations` and `exchanges`), and in 7 of 10 runs it then did not send the rep to the manager. The rule that fixed
`manager_phone` broke `discount`. A next version would say that a policy about something nearby does not cover the
question. Had `manager_phone` not improved, the same evidence (the diff, both tables, the replies) would still
answer the task: it would show the rule did not reach the step that fails. Each change moves more than one row, which is why every version runs the whole test.

## Task 2: Test on questions you did not tune on

`v3` was changed after reading the six cases' results, so it may fit them better than new questions. Two new cases,
with their key written before the run.

In [8]:
NEW_CASES = {
    "other_email": {"question": "Can the customer have the album sent to a different email address?",
                    "policy": ["digital_downloads"], "fact": ["not send", "don't send", "cannot", "can't", "unable"]},
    "three_tickets": {"question": "The same customer now has three open tickets. What should I do?",
                      "policy": ["escalations"], "fact": ["manager"]},
}
new_rows = []
for version, prompt in [("v1", v1), ("v2", v2), ("v3", v3)]:
    rows = run_tests(create_agent(model, tools=[policy_tool], system_prompt=prompt), version, cases=NEW_CASES)
    new_rows += rows
    print(version); show(rows); print()

v1
case            right_policy  has_fact   both
other_email              0/3       1/3    0/3
three_tickets            2/3       3/3    2/3
both checks: 2 of 6 runs



v2
case            right_policy  has_fact   both
other_email              0/3       2/3    0/3
three_tickets            3/3       3/3    3/3
both checks: 3 of 6 runs



v3
case            right_policy  has_fact   both
other_email              0/3       2/3    0/3
three_tickets            3/3       3/3    3/3
both checks: 3 of 6 runs



In [9]:
for row in new_rows:
    if row["case"] == "other_email":
        print(f"[{row['version']} run {row['run']}] fetched {row['fetched']}\n  {row['reply'][:200]}\n")

[v1 run 1] fetched ['delivery']
  There is no specific policy regarding sending albums to a different email address. Please consult the support manager for guidance.

[v1 run 2] fetched ['email address changes']
  There is no policy covering changes to the email address for album delivery. Please consult the support manager for guidance.

[v1 run 3] fetched ['email address changes']
  There is no specific policy regarding sending an album to a different email address, so please consult the support manager for guidance. 

For customer communication: "Unfortunately, I can't send the 

[v2 run 1] fetched ['albums']
  No. There is no specific policy regarding sending albums to a different email address. Please check with the support manager for guidance.

[v2 run 2] fetched ['accounts']
  No, the customer cannot have the album sent to a different email address because one account is tied to one email. Policy: Accounts. 

Suggested reply: "I'm sorry, but we can only send the album to th

[v

`v1` passed 2 of 6 runs, `v2` and `v3` 3 of 6 each. On the three open tickets `v2` and `v3` fetched `escalations`
and named the manager every time; `v1` fetched it in 2 of 3 runs, yet named the manager in all three, because the
trigger is written into its prompt. `other_email` failed in every version: no run fetched `digital_downloads`. `v1`
guessed names (`delivery`, `email address changes`) and said no policy covers it. `v2` and `v3` said *No* from the
accounts policy in 5 of 6 runs: the right answer from the wrong file, which `right_policy` catches and a reader would
not. (One `v3` reply, *an album can only be sent to the account's registered email address*, says no in words the
key does not list, so `has_fact` missed it: read passing and failing replies alike.) The six cases rated `v3` best;
the new questions show that its gain does not reach a question whose policy name the model cannot guess. The fix for
that is the tool's description (the lab's Going further), not the prompt.

Report the two scores separately: the six cases say how well a prompt fits the questions you tuned it on, the new
ones how well it holds on questions you did not. When they disagree, trust the new ones, and keep adding cases a
prompt was not written on.